# Bab 5: Persiapan Data

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab05/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab05")
print("Folder kerja:", os.getcwd())

## Listing 5.1: Membersihkan data anggota mentah

`kode/bab05/bersihkan.py`

In [ ]:
import numpy as np
import pandas as pd

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_mentah.csv")
print("awal          :", df.shape)

df = df.drop_duplicates()
print("tanpa duplikat:", df.shape)

KOTA = {"pekanbaru": "Pekanbaru", "pekan baru": "Pekanbaru",
        "jogja": "Yogyakarta", "yogya": "Yogyakarta",
        "diy": "Yogyakarta", "dki jakarta": "Jakarta",
        "bdg": "Bandung"}
kunci = df["kota"].str.strip().str.lower()
df["kota"] = kunci.map(KOTA).fillna(kunci.str.title())
print("kota:", df["kota"].nunique(), "nilai unik")

df["tanggal_daftar"] = pd.to_datetime(df["tanggal_daftar"],
                                      format="mixed", dayfirst=True)

teks = df["nilai_90h"].astype(str)
df["nilai_90h"] = pd.to_numeric(
    teks.str.replace("Rp", "").str.replace(".", "", regex=False))

df["umur"] = pd.to_numeric(df["umur"])
salah = ~df["umur"].between(15, 90) & df["umur"].notna()
df.loc[salah, "umur"] = np.nan
print("umur tak wajar -> NaN:", int(salah.sum()))

meter = df["jarak_km"] > 100
df.loc[meter, "jarak_km"] = df.loc[meter, "jarak_km"] / 1000
print("jarak dari meter:", int(meter.sum()))

df.to_csv(DATA + "pelanggan_bersih.csv", index=False)
acuan = pd.read_csv(DATA + "pelanggan.csv").set_index("id_anggota")
cek = df.set_index("id_anggota").loc[acuan.index]
print("nilai_90h cocok :", (cek["nilai_90h"] == acuan["nilai_90h"]).all())
print("kota cocok      :", (cek["kota"] == acuan["kota"]).all())

## Listing 5.2: Memeriksa dan mengisi umur yang kosong

`kode/bab05/hilang.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer, SimpleImputer

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv")
benar = pd.read_csv(DATA + "pelanggan.csv").set_index("id_anggota")["umur"]

kosong = df["umur"].isna()
print("umur kosong:", int(kosong.sum()))
print(df.groupby(kosong)[["churn", "frekuensi_90h"]]
      .mean().round(3).rename(index={False: "terisi", True: "kosong"})
      .to_string())

def galat(isian):
    t = benar.loc[df.loc[kosong, "id_anggota"]].values
    return np.sqrt(np.mean((isian[kosong.values] - t) ** 2))

rata = df["umur"].fillna(df["umur"].mean()).values
median = df["umur"].fillna(df["umur"].median()).values
kelompok = (df["umur"].fillna(
    df.groupby("kota")["umur"]
    .transform("median"))).values
kol = ["umur", "porsi_pagi", "porsi_akhir_pekan", "porsi_kopi_hitam",
       "rata_nilai_struk", "porsi_promo"]
X = df[kol].fillna({"porsi_aplikasi": 0})
knn = KNNImputer(n_neighbors=15).fit_transform(
    (X - X.mean()) / X.std())[:, 0] * X["umur"].std() + X["umur"].mean()

for nama, isian in [("rata-rata", rata), ("median", median),
                    ("median per kota", kelompok), ("kNN", knn)]:
    print(f"{nama:<16} RMSE = {galat(isian):.2f} tahun")
penanda = SimpleImputer(strategy="median", add_indicator=True)
print(penanda.fit_transform(df[["umur"]]).shape)

## Listing 5.3: Membandingkan transformasi nilai belanja

`kode/bab05/transformasi.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (MinMaxScaler, PowerTransformer,
                                   RobustScaler, StandardScaler)

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv")
x = df[["nilai_90h"]] / 1000

hasil = pd.DataFrame({
    "asli": x["nilai_90h"],
    "minmax": MinMaxScaler().fit_transform(x).ravel(),
    "z": StandardScaler().fit_transform(x).ravel(),
    "robust": RobustScaler().fit_transform(x).ravel(),
    "log": np.log1p(x["nilai_90h"]),
    "yeo": PowerTransformer().fit_transform(x).ravel(),
})
print(hasil.describe().loc[["mean", "std", "min", "50%", "max"]]
      .round(2).to_string())
print(hasil.skew().round(2).to_string())

## Listing 5.4: Tiga cara mendiskretisasi recency

`kode/bab05/diskretisasi.py`

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv")
r = df["recency_hari"]

lebar = pd.cut(r, bins=4)
frek = pd.qcut(r, q=4, duplicates="drop")
print(lebar.value_counts(sort=False).to_string())
print(frek.value_counts(sort=False).to_string())

pohon = DecisionTreeClassifier(max_leaf_nodes=4,
                               min_samples_leaf=300)
pohon.fit(df[["recency_hari"]], df["churn"])
ambang = sorted(t for t in pohon.tree_.threshold if t > 0)
print("ambang terbimbing:", [round(float(t), 1) for t in ambang])
kelas = pd.cut(r, [-1] + ambang + [r.max()])
print(df.groupby(kelas, observed=True)["churn"]
      .agg(["size", "mean"]).round(3).to_string())

## Listing 5.5: Mengodekan atribut kategorik

`kode/bab05/kodekan.py`

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv")

oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
kota = oh.fit_transform(df[["kota"]])
print(kota.shape, list(oh.get_feature_names_out())[:3])

urutan = [["perunggu", "perak", "emas"]]
od = OrdinalEncoder(categories=urutan)
print(od.fit_transform(df[["tingkat"]])[:5].ravel())

print(pd.get_dummies(df["jenis_kelamin"], prefix="jk",
                     dummy_na=True).head(3).astype(int).to_string())

target = df.groupby("kota")["churn"].mean()
print(target.round(3).sort_values().to_string())

## Listing 5.6: Menghitung RFM dari data transaksi

`kode/bab05/fitur_rfm.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
trx = pd.read_csv(DATA + "transaksi_lengkap_2026q1.csv",
                  usecols=["no_struk", "waktu", "id_anggota", "nilai"],
                  parse_dates=["waktu"])
trx = trx.dropna(subset=["id_anggota"])
potong = pd.Timestamp("2026-04-01")

struk = (trx.groupby(["id_anggota", "no_struk"])
         .agg(waktu=("waktu", "max"), nilai=("nilai", "sum"))
         .reset_index())
rfm = struk.groupby("id_anggota").agg(
    recency=("waktu", lambda w: (potong - w.max()).days),
    frequency=("no_struk", "nunique"),
    monetary=("nilai", "sum"))
print(rfm.shape)
print(rfm.describe().loc[["mean", "50%", "max"]].round(0).to_string())

skor = pd.DataFrame({
    "R": pd.qcut(rfm["recency"].rank(method="first"), 5,
                 labels=[5, 4, 3, 2, 1]),
    "F": pd.qcut(rfm["frequency"].rank(method="first"), 5,
                 labels=[1, 2, 3, 4, 5]),
    "M": pd.qcut(rfm["monetary"].rank(method="first"), 5,
                 labels=[1, 2, 3, 4, 5])}).astype(int)
rfm["segmen"] = skor.astype(str).sum(axis=1)
print(rfm["segmen"].value_counts().head(4).to_string())
print(rfm.loc[rfm["segmen"] == "555"].head(2).to_string())

## Listing 5.7: Seleksi fitur untuk prediksi churn

`kode/bab05/seleksi.py`

In [ ]:
import pandas as pd
from sklearn.feature_selection import (RFE, VarianceThreshold,
                                       mutual_info_classif)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv").dropna()
df["tren_frek"] = df["frekuensi_90h"] / (df["frekuensi_90h_lalu"] + 1)
fitur = ["umur", "frekuensi_90h", "tren_frek", "recency_hari",
         "nilai_90h", "rata_nilai_struk", "porsi_promo",
         "porsi_aplikasi", "porsi_kopi_hitam", "porsi_pagi",
         "porsi_akhir_pekan", "ragam_menu", "keluhan_90h",
         "jarak_km", "pakai_ulasan"]
X, y = df[fitur], df["churn"]

vt = VarianceThreshold(0.01).fit(X)
print("varians kecil:", list(X.columns[~vt.get_support()]))

mi = mutual_info_classif(X, y, random_state=0)
mi = pd.Series(mi, index=fitur).sort_values(ascending=False)
print(mi.round(3).head(6).to_string())

Xs = StandardScaler().fit_transform(X)
rfe = RFE(LogisticRegression(max_iter=1000),
          n_features_to_select=6).fit(Xs, y)
import textwrap
print(textwrap.fill("RFE: " + ", ".join(X.columns[rfe.support_]), 58))

## Listing 5.8: Membandingkan cara menangani data tidak seimbang

`kode/bab05/seimbang.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv").dropna()
fitur = ["frekuensi_90h", "recency_hari", "porsi_promo",
         "keluhan_90h", "jarak_km", "nilai_90h"]
X = StandardScaler().fit_transform(df[fitur])
y = df["churn"].values
print("porsi churn:", y.mean().round(3))
Xl, Xu, yl, yu = train_test_split(X, y, test_size=.3,
                                  stratify=y, random_state=1)

def smote(X, y, k=5, rng=np.random.default_rng(0)):
    Xm = X[y == 1]
    perlu = (y == 0).sum() - len(Xm)
    jarak = ((Xm[:, None, :] - Xm[None, :, :]) ** 2).sum(-1)
    tetangga = np.argsort(jarak, axis=1)[:, 1:k + 1]
    i = rng.integers(0, len(Xm), perlu)
    j = tetangga[i, rng.integers(0, k, perlu)]
    lam = rng.random((perlu, 1))
    baru = Xm[i] + lam * (Xm[j] - Xm[i])
    return np.vstack([X, baru]), np.r_[y, np.ones(perlu, int)]

rng = np.random.default_rng(0)
idx0, idx1 = np.where(yl == 0)[0], np.where(yl == 1)[0]
bawah = np.r_[rng.choice(idx0, len(idx1), replace=False), idx1]
atas = np.r_[idx0, rng.choice(idx1, len(idx0), replace=True)]
cara = {"asli": (Xl, yl, None), "undersampling": (Xl[bawah], yl[bawah], None),
        "oversampling": (Xl[atas], yl[atas], None),
        "SMOTE": (*smote(Xl, yl), None),
        "bobot kelas": (Xl, yl, "balanced")}
for nama, (Xa, ya, bobot) in cara.items():
    m = LogisticRegression(class_weight=bobot, max_iter=1000).fit(Xa, ya)
    p = m.predict(Xu)
    print(f"{nama:<14} n={len(ya):>5}  presisi={precision_score(yu, p):.2f}"
          f"  recall={recall_score(yu, p):.2f}")

## Listing 5.9: Fitur yang membocorkan masa depan

`kode/bab05/bocor.py`

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv")
fitur = ["frekuensi_90h", "recency_hari", "porsi_promo", "keluhan_90h"]
model = make_pipeline(StandardScaler(), LogisticRegression())

auc = cross_val_score(model, df[fitur], df["churn"], cv=5,
                      scoring="roc_auc").mean()
print(f"fitur sah          : AUC = {auc:.3f}")

df["belanja_depan_ada"] = (df["belanja_bulan_depan"] > 0).astype(int)
bocor = fitur + ["belanja_depan_ada"]
auc = cross_val_score(model, df[bocor], df["churn"], cv=5,
                      scoring="roc_auc").mean()
print(f"dengan fitur bocor : AUC = {auc:.3f}")

## Listing 5.10: Pipeline praproses dan model churn

`kode/bab05/pipa.py`

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan_bersih.csv")
numerik = ["umur", "frekuensi_90h", "recency_hari", "nilai_90h",
           "porsi_promo", "porsi_aplikasi", "keluhan_90h", "jarak_km"]
kategorik = ["kota", "jenis_kelamin"]

praproses = ColumnTransformer([
    ("num", Pipeline([("isi", SimpleImputer(strategy="median",
                                            add_indicator=True)),
                      ("skala", StandardScaler())]), numerik),
    ("kat", Pipeline([("isi", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]),
     kategorik)])
pipa = Pipeline([("praproses", praproses),
                 ("model", LogisticRegression(max_iter=1000,
                                              class_weight="balanced"))])

skor = cross_val_score(pipa, df[numerik + kategorik], df["churn"],
                       cv=5, scoring="roc_auc")
print("AUC per lipatan:", skor.round(3))
print("rata-rata      :", skor.mean().round(3))
pipa.fit(df[numerik + kategorik], df["churn"])
print(pipa.named_steps["praproses"].transform(
    df[numerik + kategorik]).shape)